# Phase 3 - Step 13: Skill Gap Engine

This notebook calculates employee-level skill gaps, matched skills, missing skills, skill gap percentages, and readiness scores using set theory operations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

processed_dir = Path("data/processed")

df_emp_skills = pd.read_csv(processed_dir / "employee_skills.csv")
df_role_skills = pd.read_csv(processed_dir / "role_skill_matrix.csv")

# Role required skills dictionary
role_required_map = df_role_skills.groupby('job_role')['required_skill'].apply(lambda x: set(x.str.strip().str.lower())).to_dict()

# Employee current skills dictionary
emp_current_map = df_emp_skills.groupby(['employee_id', 'job_role', 'department', 'name'])['skill_name'].apply(
    lambda x: set(x.str.strip().str.lower())
).reset_index()

gap_records = []

for idx, row in emp_current_map.iterrows():
    emp_id = row['employee_id']
    role = row['job_role']
    dept = row['department']
    name = row['name']
    curr_skills = row['skill_name']
    
    # Required skills for role
    req_skills = role_required_map.get(role, set(['problem solving', 'communication']))
    
    # Set operations
    matched = req_skills.intersection(curr_skills)
    missing = req_skills - curr_skills
    
    total_req = len(req_skills)
    total_matched = len(matched)
    
    gap_pct = round((len(missing) / total_req) * 100.0, 2) if total_req > 0 else 0.0
    readiness = round((total_matched / total_req) * 100.0, 2) if total_req > 0 else 100.0
    
    # Capitalize for display
    matched_display = [s.title() for s in sorted(list(matched))]
    missing_display = [s.title() for s in sorted(list(missing))]
    
    gap_records.append({
        "employee_id": emp_id,
        "name": name,
        "department": dept,
        "job_role": role,
        "matched_skills": "; ".join(matched_display),
        "missing_skills": "; ".join(missing_display),
        "total_required_skills": total_req,
        "total_matched_skills": total_matched,
        "missing_skills_count": len(missing),
        "skill_gap_percentage": gap_pct,
        "readiness_score": readiness
    })

df_gaps = pd.DataFrame(gap_records)
df_gaps.to_csv(processed_dir / "employee_skill_gaps.csv", index=False)
print(f"Generated employee_skill_gaps.csv for {len(df_gaps)} employees.")
print(f"Average Readiness Score: {df_gaps['readiness_score'].mean():.2f}%")
print(f"Average Skill Gap: {df_gaps['skill_gap_percentage'].mean():.2f}%")
display(df_gaps.head(5))


Generated employee_skill_gaps.csv for 5500 employees.
Average Readiness Score: 68.21%
Average Skill Gap: 31.79%


,employee_id,name,department,job_role,matched_skills,missing_skills,total_required_skills,total_matched_skills,missing_skills_count,skill_gap_percentage,readiness_score
0,1,Steven Barnett,Finance,Auditor,Compliance; Excel; Internal Auditing; Risk Ass...,Financial Analysis; Forensic Accounting,6,4,2,33.33,66.67
1,2,Christopher Benson,Sales,Sales Executive,Communication; Contract Closing; Crm; Lead Gen...,B2B Sales,6,5,1,16.67,83.33
2,3,Norman Lane,Support,Helpdesk,Active Directory; Customer Service; Hardware S...,Windows Os,5,4,1,20.00,80.00
3,4,Rita Walker,HR,HR Executive,Communication; Hris; Onboarding; Recruitment,Hr Operations,5,4,1,20.00,80.00
4,5,Judith Ware,Sales,Account Manager,Account Planning; Client Retention; Contract N...,Crm (Salesforce); Upselling,5,3,2,40.00,60.00
